In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import os

# ── reproducibility ─────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# =============================================================================
# 1.  CUSTOM DATASET  (tech + sports domain)
# =============================================================================

sentences = [
    "Barack Obama was born in Hawaii",
    "Google is based in Mountain View",
    "Elon Musk founded SpaceX in California",
    "Microsoft is headquartered in Redmond",
    "Cristiano Ronaldo plays for Al Nassr in Riyadh",
    "Amazon was founded by Jeff Bezos in Seattle",
    "Lionel Messi joined Inter Miami in Florida",
    "Apple released the iPhone in Cupertino",
    "Tim Cook leads Apple in Cupertino California",
    "Sundar Pichai runs Google in Mountain View",
]

labels = [
    ["PERSON",       "PERSON", "O",    "O",     "O",  "LOCATION"],
    ["ORGANIZATION", "O",      "O",    "O",     "LOCATION", "LOCATION"],
    ["PERSON",       "PERSON", "O",    "ORGANIZATION", "O", "LOCATION"],
    ["ORGANIZATION", "O",      "O",    "O",     "LOCATION"],
    ["PERSON",       "PERSON", "O",    "O",     "ORGANIZATION", "ORGANIZATION", "O", "LOCATION"],
    ["ORGANIZATION", "O",      "O",    "O",     "PERSON", "PERSON", "O", "LOCATION"],
    ["PERSON",       "PERSON", "O",    "ORGANIZATION", "ORGANIZATION", "O", "LOCATION"],
    ["ORGANIZATION", "O",      "O",    "O",     "O",  "LOCATION"],
    ["PERSON",       "PERSON", "O",    "ORGANIZATION", "O", "LOCATION", "LOCATION"],
    ["PERSON",       "PERSON", "O",    "ORGANIZATION", "O", "LOCATION", "LOCATION"],
]

# =============================================================================
# 2.  VOCABULARIES
# =============================================================================

PAD_TOKEN  = "<PAD>"
UNK_TOKEN  = "<UNK>"

# build word vocab
word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for sent in sentences:
    for w in sent.lower().split():
        if w not in word2idx:
            word2idx[w] = len(word2idx)
idx2word = {v: k for k, v in word2idx.items()}

# build label vocab
label_set   = ["PAD", "O", "PERSON", "LOCATION", "ORGANIZATION"]
label2idx   = {l: i for i, l in enumerate(label_set)}
idx2label   = {v: k for k, v in label2idx.items()}

PAD_LABEL_IDX = label2idx["PAD"]
VOCAB_SIZE    = len(word2idx)
NUM_CLASSES   = len(label2idx)

print("=" * 60)
print("  RNN NER — PyTorch")
print("=" * 60)
print(f"  Vocabulary size : {VOCAB_SIZE}")
print(f"  Label classes   : {label_set}")
print(f"  Training samples: {len(sentences)}")

# =============================================================================
# 3.  ENCODE & PAD
# =============================================================================

MAX_LEN = max(len(s.split()) for s in sentences)

def encode_sentence(sent):
    return [word2idx.get(w.lower(), word2idx[UNK_TOKEN]) for w in sent.split()]

def encode_labels(lbls):
    return [label2idx[l] for l in lbls]

def pad(seq, max_len, pad_val):
    return seq + [pad_val] * (max_len - len(seq))

X_data = [pad(encode_sentence(s), MAX_LEN, word2idx[PAD_TOKEN]) for s in sentences]
y_data = [pad(encode_labels(l),   MAX_LEN, PAD_LABEL_IDX)       for l in labels]

X_tensor = torch.tensor(X_data, dtype=torch.long)   # (N, MAX_LEN)
y_tensor = torch.tensor(y_data, dtype=torch.long)    # (N, MAX_LEN)

print(f"  X shape         : {tuple(X_tensor.shape)}")
print(f"  y shape         : {tuple(y_tensor.shape)}\n")

# =============================================================================
# 4.  DATASET & DATALOADER
# =============================================================================

class NERDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset    = NERDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# =============================================================================
# 5.  RNN MODEL
# =============================================================================

class RNN_NER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_units, num_classes,
                 dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim,
                                      padding_idx=word2idx[PAD_TOKEN])
        self.rnn       = nn.RNN(embed_dim, hidden_units,
                                batch_first=True, nonlinearity='tanh')
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_units, num_classes)

    def forward(self, x):
        emb    = self.embedding(x)          # (B, T, embed_dim)
        out, _ = self.rnn(emb)              # (B, T, hidden_units)
        out    = self.dropout(out)
        logits = self.fc(out)               # (B, T, num_classes)
        return logits

# =============================================================================
# 6.  TRAINING FUNCTION
# =============================================================================

def train_model(units, epochs, lr, label):
    model     = RNN_NER(VOCAB_SIZE, embed_dim=50,
                        hidden_units=units, num_classes=NUM_CLASSES)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_LABEL_IDX)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    acc_history  = []
    loss_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            logits = model(X_batch)                      # (B, T, C)
            # reshape for loss: (B*T, C) vs (B*T,)
            loss   = criterion(logits.view(-1, NUM_CLASSES), y_batch.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # accuracy (ignore PAD)
            preds = logits.argmax(dim=-1)                # (B, T)
            mask  = y_batch != PAD_LABEL_IDX
            correct += (preds[mask] == y_batch[mask]).sum().item()
            total   += mask.sum().item()

        acc  = correct / total if total > 0 else 0
        acc_history.append(acc)
        loss_history.append(total_loss / len(dataloader))

        if epoch % max(1, epochs // 5) == 0 or epoch == epochs:
            print(f"  [{label}] Epoch {epoch:>3}/{epochs} | "
                  f"Loss: {loss_history[-1]:.4f} | Acc: {acc:.4f}")

    print()
    return model, acc_history, loss_history

# =============================================================================
# 7.  EXPERIMENT — vary units, epochs, learning rate  (Task requirement)
# =============================================================================

configs = [
    {"units": 32,  "epochs": 15, "lr": 0.01,   "label": "Small  (32u, lr=1e-2)"},
    {"units": 64,  "epochs": 25, "lr": 0.001,  "label": "Medium (64u, lr=1e-3)"},
    {"units": 128, "epochs": 40, "lr": 0.0005, "label": "Large  (128u, lr=5e-4)"},
]

results = {}
for cfg in configs:
    print("─"*60)
    print(f"  Config: {cfg['label']}")
    print("─"*60)
    model, acc_hist, loss_hist = train_model(
        units=cfg["units"], epochs=cfg["epochs"],
        lr=cfg["lr"], label=cfg["label"]
    )
    results[cfg["label"]] = {
        "model":    model,
        "acc":      acc_hist,
        "loss":     loss_hist,
        "epochs":   cfg["epochs"],
    }

# =============================================================================
# 8.  PLOT 1 — Training Accuracy Comparison
# =============================================================================

# Create the output directory if it doesn't exist
os.makedirs("/mnt/user-data/outputs", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#e74c3c", "#2ecc71", "#3498db"]

for (label, res), color in zip(results.items(), colors):
    axes[0].plot(range(1, res["epochs"] + 1), res["acc"],
                 label=label, color=color, linewidth=2)
    axes[1].plot(range(1, res["epochs"] + 1), res["loss"],
                 label=label, color=color, linewidth=2, linestyle="--")

axes[0].set_title("Training Accuracy — Config Comparison", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

axes[1].set_title("Training Loss — Config Comparison", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/training_curves.png", dpi=150)
plt.close()
print("Saved ▶  training_curves.png\n")

# =============================================================================
# 9.  PREDICTION HELPER
# =============================================================================

def predict(model, sentence):
    model.eval()
    tokens = sentence.lower().split()
    ids    = [word2idx.get(t, word2idx[UNK_TOKEN]) for t in tokens]
    padded = ids + [word2idx[PAD_TOKEN]] * (MAX_LEN - len(ids))
    x      = torch.tensor([padded], dtype=torch.long)
    with torch.no_grad():
        logits = model(x)                                # (1, T, C)
        probs  = torch.softmax(logits, dim=-1)           # (1, T, C)
        preds  = logits.argmax(dim=-1)[0]                # (T,)
    pred_labels = [idx2label[p.item()] for p in preds[:len(tokens)]]
    prob_matrix = probs[0, :len(tokens), :].numpy()      # (words, classes)
    return tokens, pred_labels, prob_matrix

# =============================================================================
# 10.  TEST ON NEW SENTENCES
# =============================================================================

best_model = results["Large  (128u, lr=5e-4)"]["model"]

test_sentences = [
    "Elon Musk visited Seattle",
    "Apple is headquartered in Cupertino",
    "Ronaldo scored in Riyadh",
    "Jeff Bezos founded Amazon in Seattle",
    "Messi plays for Inter Miami in Florida",
]

print("=" * 60)
print("  PREDICTIONS ON NEW SENTENCES  (best model)")
print("=" * 60)

all_results = []
for sent in test_sentences:
    tokens, pred_labels, prob_matrix = predict(best_model, sent)
    all_results.append((tokens, pred_labels, prob_matrix))
    print(f"\n  Sentence : \"{sent}\"")
    print(f"  {'Word':<20} {'Predicted Label'}")
    print("  " + "─" * 35)
    for word, lbl in zip(tokens, pred_labels):
        tag = f"[{lbl}]" if lbl != "O" else ""
        print(f"  {word:<20} {lbl}  {tag}")

# =============================================================================
# 11.  PLOT 2 — Prediction Probability Heatmaps (first 3 test sentences)
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
label_names = [idx2label[i] for i in range(NUM_CLASSES)]

for ax, (tokens, pred_labels, prob_matrix) in zip(axes, all_results[:3]):
    im = ax.imshow(prob_matrix.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(label_names, fontsize=9)
    ax.set_title(" ".join(tokens), fontsize=9, fontweight="bold")

    # annotate cells with probability
    for i in range(len(tokens)):
        for j in range(NUM_CLASSES):
            val = prob_matrix[i, j]
            ax.text(i, j, f"{val:.2f}", ha='center', va='center',
                    fontsize=7, color='black' if val < 0.7 else 'white')

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="P")

fig.suptitle("NER Prediction Probability Heatmaps", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/prediction_heatmaps.png", dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved ▶  prediction_heatmaps.png\n")

# =============================================================================
# 12.  PLOT 3 — RNN Architecture Diagram
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 6); ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')
ax.set_title("Simple RNN Architecture for NER", fontsize=14,
             fontweight='bold', pad=15)

layer_colors = {
    "Embedding"   : "#3498db",
    "SimpleRNN"   : "#e74c3c",
    "Dropout"     : "#95a5a6",
    "Linear (FC)" : "#2ecc71",
    "Softmax"     : "#f39c12",
}
layers = list(layer_colors.items())

positions = [1.2, 3.0, 4.8, 6.6, 8.4]
for (name, color), x in zip(layers, positions):
    rect = mpatches.FancyBboxPatch((x - 0.8, 2.0), 1.6, 1.8,
        boxstyle="round,pad=0.15", linewidth=2,
        edgecolor='white', facecolor=color, alpha=0.88)
    ax.add_patch(rect)
    ax.text(x, 3.2, name, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

# arrows between layers
for i in range(len(positions) - 1):
    ax.annotate("", xy=(positions[i+1] - 0.85, 2.9),
                xytext=(positions[i] + 0.85, 2.9),
                arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=2))

# input / output labels
ax.text(positions[0],  1.4, "Word IDs\n(padded)", ha='center', fontsize=8,
        color='#2c3e50', style='italic')
ax.text(positions[-1], 1.4, "NER Labels\n(per token)", ha='center', fontsize=8,
        color='#2c3e50', style='italic')

# dim annotations
dims = ["(B, T) → (B, T, 50)", "(B, T, 50) → (B, T, H)",
        "drop 20%", "(B, T, H) → (B, T, C)", "probabilities"]
for dim, x in zip(dims, positions):
    ax.text(x, 4.15, dim, ha='center', fontsize=7, color='#555')

# legend for final classes
ax.text(10.2, 4.6, "Output Classes:", fontsize=9, fontweight='bold', color='#2c3e50')
class_colors = {"O":"#bdc3c7", "PERSON":"#9b59b6",
                "LOCATION":"#1abc9c", "ORGANIZATION":"#e67e22"}
for i, (cls, c) in enumerate(class_colors.items()):
    rect = mpatches.FancyBboxPatch((10.0, 3.8 - i * 0.55), 1.8, 0.45,
        boxstyle="round,pad=0.05", facecolor=c, edgecolor='white', alpha=0.85)
    ax.add_patch(rect)
    ax.text(10.9, 4.02 - i * 0.55, cls, ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/rnn_architecture.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved ▶  rnn_architecture.png\n")

# =============================================================================
# 13.  SUMMARY
# =============================================================================

print("=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
for label, res in results.items():
    print(f"  {label}")
    print(f"    Best Accuracy : {max(res['acc']):.4f}")
    print(f"    Final Loss    : {res['loss'][-1]:.4f}")
print()
print("  Output files:")
print("    ✔  training_curves.png")
print("    ✔  prediction_heatmaps.png")
print("    ✔  rnn_architecture.png")
print("=" * 60)


  RNN NER — PyTorch
  Vocabulary size : 50
  Label classes   : ['PAD', 'O', 'PERSON', 'LOCATION', 'ORGANIZATION']
  Training samples: 10
  X shape         : (10, 8)
  y shape         : (10, 8)

────────────────────────────────────────────────────────────
  Config: Small  (32u, lr=1e-2)
────────────────────────────────────────────────────────────
  [Small  (32u, lr=1e-2)] Epoch   3/15 | Loss: 0.7519 | Acc: 0.7576
  [Small  (32u, lr=1e-2)] Epoch   6/15 | Loss: 0.2033 | Acc: 1.0000
  [Small  (32u, lr=1e-2)] Epoch   9/15 | Loss: 0.0411 | Acc: 1.0000
  [Small  (32u, lr=1e-2)] Epoch  12/15 | Loss: 0.0161 | Acc: 1.0000
  [Small  (32u, lr=1e-2)] Epoch  15/15 | Loss: 0.0113 | Acc: 1.0000

────────────────────────────────────────────────────────────
  Config: Medium (64u, lr=1e-3)
────────────────────────────────────────────────────────────
  [Medium (64u, lr=1e-3)] Epoch   5/25 | Loss: 1.0059 | Acc: 0.8485
  [Medium (64u, lr=1e-3)] Epoch  10/25 | Loss: 0.5903 | Acc: 0.9242
  [Medium (64u, lr=1e